
1. orders.csv-ni EXPLICIT schema ilə, shipments.json-u isə birbaşa
   (nested sxem avtomatik tanınmalıdır) oxu.

In [5]:
import os
import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType
)
# Python versiya uyğunsuzluğunu qarşısını almaq üçün mühit dəyişənləri
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 1. SparkSession-ın yaradılması
spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [7]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("amount", DoubleType(), True),
    StructField("region", StringType(), True)
])

df_orders = (
    spark.read
    .schema(orders_schema)
    .option("header", "true")
    .csv("/home/jovyan/work/orders.csv")
)

df_shipments = (
    spark.read
    .option("multiline", "true")
    .json("/home/jovyan/work/shipments.json")
)

df_orders.show(3)
df_shipments.show(3, truncate=False)

+--------+-----------+----------------+----------+------+--------+
|order_id|customer_id|   customer_name|order_date|amount|  region|
+--------+-----------+----------------+----------+------+--------+
|   O0001|      CU012|  Ilkin Karimova|2026-03-01|318.46|   Gəncə|
|   O0002|      CU001|  Tural Karimova|2026-03-21|360.87|    Bakı|
|   O0003|      CU002|Ilkin Ismayilova|2026-01-11|228.44|Sumqayıt|
+--------+-----------+----------------+----------+------+--------+
only showing top 3 rows

+------------------------------------+--------+
|items                               |order_id|
+------------------------------------+--------+
|[{0.8, Pen, 2}, {1.5, Snack Bar, 2}]|O0001   |
+------------------------------------+--------+



2. orders üzərində: amount NULL olan sətirləri 0 ilə əvəz et;
   amount-a görə order_size sütunu yarat (< 50 "small", 50-200
   "medium", 200+ "large").

In [8]:
df_orders_clean = (
    df_orders
    .fillna({"amount": 0.0})
    .withColumn(
        "order_size",
        F.when(F.col("amount") < 50, "small")
         .when((F.col("amount") >= 50) & (F.col("amount") <= 200), "medium")
         .otherwise("large")
    )
    .withColumn("order_month", F.date_format(F.col("order_date"), "yyyy-MM"))
)

df_orders_clean.show(5)

+--------+-----------+----------------+----------+------+--------+----------+-----------+
|order_id|customer_id|   customer_name|order_date|amount|  region|order_size|order_month|
+--------+-----------+----------------+----------+------+--------+----------+-----------+
|   O0001|      CU012|  Ilkin Karimova|2026-03-01|318.46|   Gəncə|     large|    2026-03|
|   O0002|      CU001|  Tural Karimova|2026-03-21|360.87|    Bakı|     large|    2026-03|
|   O0003|      CU002|Ilkin Ismayilova|2026-01-11|228.44|Sumqayıt|     large|    2026-01|
|   O0004|      CU012|  Ilkin Karimova|2026-06-25|204.62|    Bakı|     large|    2026-06|
|   O0005|      CU003|   Murad Guliyev|2026-01-20|   0.0|   Gəncə|     small|    2026-01|
+--------+-----------+----------------+----------+------+--------+----------+-----------+
only showing top 5 rows



3. orders və shipments-i order_id üzrə INNER join et; ayrıca, heç
   bir shipment qeydi olmayan sifarişləri (LEFT ANTI) tap.


In [9]:
df_inner_joined = df_orders_clean.join(df_shipments, on="order_id", how="inner")
df_no_shipment = df_orders_clean.join(df_shipments, on="order_id", how="left_anti")
df_inner_joined.count(), df_no_shipment.count()

(1, 29)

4. Window function ilə hər customer_id üçün ən böyük 3 sifarişi
   (amount-a görə) seç.

In [10]:
df_orders_clean.createOrReplaceTempView("orders_clean")

df_top3_orders_sql = spark.sql("""
    WITH ranked_orders AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) as rn
        FROM orders_clean
    )
    SELECT * 
    FROM ranked_orders 
    WHERE rn <= 3
""")

df_top3_orders_sql.show(10)

+--------+-----------+----------------+----------+------+--------+----------+-----------+---+
|order_id|customer_id|   customer_name|order_date|amount|  region|order_size|order_month| rn|
+--------+-----------+----------------+----------+------+--------+----------+-----------+---+
|   O0030|      CU001|  Tural Karimova|2026-04-19|433.74|   Gəncə|     large|    2026-04|  1|
|   O0002|      CU001|  Tural Karimova|2026-03-21|360.87|    Bakı|     large|    2026-03|  2|
|   O0024|      CU001|  Tural Karimova|2026-05-28|  44.0|   Gəncə|     small|    2026-05|  3|
|   O0017|      CU002|Ilkin Ismayilova|2026-03-31|377.43|   Gəncə|     large|    2026-03|  1|
|   O0003|      CU002|Ilkin Ismayilova|2026-01-11|228.44|Sumqayıt|     large|    2026-01|  2|
|   O0018|      CU003|   Murad Guliyev|2026-01-19|330.91|Sumqayıt|     large|    2026-01|  1|
|   O0026|      CU003|   Murad Guliyev|2026-07-21| 51.95|   Gəncə|    medium|    2026-07|  2|
|   O0012|      CU003|   Murad Guliyev|2026-09-11|  21.3|   

5. region üzrə qruplaşdırıb, cəmi amount və sifariş sayını hesabla,
   cəmi amount-a görə azalan sırala.

In [14]:
df_region_summary_sql = spark.sql("""
    SELECT 
        region,
        SUM(amount) AS total_amount,
        COUNT(order_id) AS order_count
    FROM orders_clean
    group BY region
    order BY total_amount DESC
""")


 6. shipments-dəki items array-ını explode et, hər item üçün
   line_total (qty × price) hesabla, order_id üzrə cəmləyib
   orders.amount ilə müqayisə et.


In [12]:
df_exploded = (
    df_inner_joined
    .withColumn("item", F.explode("items"))
    .select(
        "order_id",
        "amount",
        F.col("item.product").alias("product"),
        F.col("item.qty").alias("qty"),
        F.col("item.price").alias("price")
    )
    .withColumn("line_total", F.col("qty") * F.col("price"))
)

df_comparison = (
    df_exploded
    .groupBy("order_id", "amount")
    .agg(F.sum("line_total").alias("calculated_total"))
    .withColumn("difference",F.col("amount") - F.col("calculated_total"))
)

df_comparison.show(5)

+--------+------+----------------+------------------+
|order_id|amount|calculated_total|        difference|
+--------+------+----------------+------------------+
|   O0001|318.46|             4.6|313.85999999999996|
+--------+------+----------------+------------------+




7. customer_id üzrə qruplaşdırıb, order_date-dən çıxarılan aya görə
   PIVOT et (hər ay üçün cəmi amount sütunu).

In [13]:
df_pivot = (
    df_orders_clean
    .groupBy("customer_id")
    .pivot("order_month")
    .agg(F.sum("amount"))
    .na.fill(0)
)

df_pivot.show()

+-----------+-------+-------+-----------------+-------+-------+-------+-------+-------+
|customer_id|2026-01|2026-02|          2026-03|2026-04|2026-05|2026-06|2026-07|2026-09|
+-----------+-------+-------+-----------------+-------+-------+-------+-------+-------+
|      CU014|    0.0|    0.0|              0.0|    0.0|  54.33|    0.0|    0.0|    0.0|
|      CU012| 426.26|    0.0|           318.46|    0.0|    0.0| 204.62|    0.0|  42.72|
|      CU006|    0.0|    0.0|              0.0|    0.0|    0.0| 133.05|    0.0|    0.0|
|      CU004|    0.0|  76.34|              0.0|    0.0|    0.0|    0.0| 442.18|    0.0|
|      CU007|    0.0|    0.0|              0.0|    0.0|  32.34|    0.0|    0.0|    0.0|
|      CU002| 228.44|    0.0|           377.43|    0.0|    0.0|    0.0|    0.0|    0.0|
|      CU013|    0.0|    0.0|              0.0|    0.0|    0.0|    0.0|  376.0|    0.0|
|      CU015|    0.0|    0.0|              0.0|    0.0|    0.0|    0.0| 154.31|    0.0|
|      CU009|    0.0|    0.0|   

8. Yekun region-üzrə xülasəni s3a://matrix/silver/region_summary
   yoluna parquet formatında, overwriteSchema=true ilə yaz.

In [16]:
(
    df_region_summary_sql
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .parquet("s3a://matrix/silver/region_summary")
)

In [17]:
spark.stop()